# 🎮 Member 4 — ML Segmentation & Dashboard
**Steam Hidden Gems Project**

This notebook covers:
1. Install dependencies
2. Load & explore the cleaned data
3. ML Clustering (K-Means)
4. Cluster interpretation & visualization
5. Export results
6. Launch the Dash dashboard

**Just run each cell top to bottom! 🚀**

---
## 1. Install Dependencies

In [ ]:
!pip install pandas scikit-learn plotly dash matplotlib seaborn -q

---
## 2. Load & Explore Data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports loaded!")

In [ ]:
# Load data — tries consolidated first, then cleaned, then sample
from pathlib import Path
import os

# Auto-detect: works whether you run from the repo root or this folder
candidates = [
    'cleaned_data/steam_consolidated.csv',
    'cleaned_data/steam_cleaned.csv',
    '../cleaned_data/steam_consolidated.csv',
    '../cleaned_data/steam_cleaned.csv',
    'samples/steam_sample.csv',
    '../samples/steam_sample.csv',
    'steam_sample.csv',  # if running from this delivery folder
]

df = None
for path in candidates:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ Loaded: {path}")
        break

if df is None:
    raise FileNotFoundError("No data file found! Place steam_sample.csv in this folder.")

print(f"📊 Shape: {df.shape[0]} games × {df.shape[1]} columns")
df.head()

In [ ]:
# Quick overview
print("=" * 50)
print("COLUMN TYPES")
print("=" * 50)
print(df.dtypes)
print()
print("=" * 50)
print("MISSING VALUES")
print("=" * 50)
print(df.isnull().sum())
print()
print("=" * 50)
print("BASIC STATS")
print("=" * 50)
df[['price', 'total_reviews', 'satisfaction_pct', 'peak_ccu']].describe().round(2)

In [ ]:
# How many games have reviews?
has_reviews = (df['total_reviews'] > 0).sum()
no_reviews = (df['total_reviews'] == 0).sum()
print(f"Games with reviews:    {has_reviews}")
print(f"Games without reviews: {no_reviews}")
print(f"Percentage reviewed:   {has_reviews / len(df) * 100:.1f}%")

---
## 3. ML Clustering — K-Means

**Objective:** Segment games into groups based on their market profile.

**Features used:**
- `price` — game pricing
- `total_reviews` — audience size (log-transformed)
- `satisfaction_pct` — player satisfaction
- `peak_ccu` — peak concurrent users (log-transformed)

**Why K-Means?** Simple, interpretable, and works well for market segmentation.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# --- Step 1: Select eligible games (need at least 1 review) ---
FEATURES = ['price', 'total_reviews', 'satisfaction_pct', 'peak_ccu']
MIN_REVIEWS = 1

eligible = df[df['total_reviews'] >= MIN_REVIEWS].copy()
print(f"Eligible games (>= {MIN_REVIEWS} review): {len(eligible)} / {len(df)}")

# --- Step 2: Prepare features ---
X = eligible[FEATURES].copy()
X = X.fillna(0)

# Log-transform skewed features
X['total_reviews'] = np.log1p(X['total_reviews'])
X['peak_ccu'] = np.log1p(X['peak_ccu'])

# Scale to zero mean, unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print("\nScaled feature stats:")
pd.DataFrame(X_scaled, columns=FEATURES).describe().round(2)

### 3.1 Finding the Best Number of Clusters
We test k = 2 to 6 and use **silhouette score** (higher = better separated clusters) and the **elbow method** (inertia).

In [ ]:
# --- Step 3: Evaluate different k values ---
K_RANGE = range(2, 7)
RANDOM_STATE = 42

results = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    inertia = km.inertia_
    sil = silhouette_score(X_scaled, labels)
    results.append({'k': k, 'inertia': inertia, 'silhouette': sil})
    print(f"  k={k}  |  inertia={inertia:,.0f}  |  silhouette={sil:.3f}")

eval_df = pd.DataFrame(results)
best_k = int(eval_df.loc[eval_df['silhouette'].idxmax(), 'k'])
print(f"\n🏆 Best k by silhouette score: {best_k}")

In [ ]:
# --- Visualize: Elbow & Silhouette ---
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score'])

fig.add_trace(
    go.Scatter(x=eval_df['k'], y=eval_df['inertia'], mode='lines+markers',
               marker=dict(size=10), line=dict(width=2), name='Inertia'),
    row=1, col=1
)

colors = ['red' if k == best_k else 'steelblue' for k in eval_df['k']]
fig.add_trace(
    go.Bar(x=eval_df['k'], y=eval_df['silhouette'], name='Silhouette',
           marker_color=colors),
    row=1, col=2
)

fig.update_layout(height=400, title_text=f'Cluster Evaluation (best k = {best_k})',
                  showlegend=False)
fig.update_xaxes(title_text='k', dtick=1)
fig.show()

### 3.2 Fit Final Model & Assign Clusters

In [ ]:
# --- Step 4: Fit final model with best k ---
final_model = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
eligible['ml_cluster'] = final_model.fit_predict(X_scaled)

print(f"✅ Assigned {best_k} clusters to {len(eligible)} games")
print(f"\nGames per cluster:")
print(eligible['ml_cluster'].value_counts().sort_index())

---
## 4. Cluster Interpretation & Visualization

In [ ]:
# --- Cluster profiles (median values) ---
profile = eligible.groupby('ml_cluster')[FEATURES].median().round(2)
print("Cluster Profiles (median values):")
print("=" * 60)
print(profile.to_string())
print()

# Human-readable labels
for cid in sorted(eligible['ml_cluster'].unique()):
    row = profile.loc[cid]
    parts = []
    if row['price'] == 0:
        parts.append('Free')
    elif row['price'] >= 15:
        parts.append('Premium')
    else:
        parts.append('Budget')
    parts.append('Popular' if row['total_reviews'] >= 100 else 'Niche')
    parts.append('High-rated' if row['satisfaction_pct'] >= 80 else ('Mixed' if row['satisfaction_pct'] >= 60 else 'Low-rated'))
    count = (eligible['ml_cluster'] == cid).sum()
    print(f"  Cluster {cid}: {' / '.join(parts)} ({count} games)")

In [ ]:
# --- Scatter: Price vs Satisfaction by Cluster ---
plot_df = eligible.copy()
plot_df['ml_cluster'] = plot_df['ml_cluster'].astype(str)

fig = px.scatter(
    plot_df, x='price', y='satisfaction_pct',
    color='ml_cluster',
    size='total_reviews',
    hover_name='name',
    title='Game Segments: Price vs Satisfaction',
    labels={'price': 'Price ($)', 'satisfaction_pct': 'Satisfaction (%)',
            'ml_cluster': 'Cluster', 'total_reviews': 'Total Reviews'},
    height=500,
)
fig.show()

In [ ]:
# --- Box plots: feature distributions per cluster ---
fig = make_subplots(rows=2, cols=2, subplot_titles=FEATURES)

for i, feat in enumerate(FEATURES):
    row = i // 2 + 1
    col = i % 2 + 1
    for cid in sorted(eligible['ml_cluster'].unique()):
        vals = eligible[eligible['ml_cluster'] == cid][feat]
        fig.add_trace(
            go.Box(y=vals, name=f'Cluster {cid}', showlegend=(i == 0)),
            row=row, col=col
        )

fig.update_layout(height=600, title_text='Feature Distributions by Cluster')
fig.show()

In [ ]:
# --- Cluster size pie chart ---
sizes = eligible['ml_cluster'].value_counts().sort_index()
fig = px.pie(
    values=sizes.values,
    names=[f'Cluster {i}' for i in sizes.index],
    title='Cluster Distribution',
    hole=0.4,
)
fig.show()

In [ ]:
# --- Sample games from each cluster ---
print("Sample games per cluster:")
print("=" * 70)
for cid in sorted(eligible['ml_cluster'].unique()):
    cluster_games = eligible[eligible['ml_cluster'] == cid]
    sample = cluster_games.nlargest(5, 'total_reviews')[['name', 'price', 'total_reviews', 'satisfaction_pct']]
    print(f"\n--- Cluster {cid} (top 5 by reviews) ---")
    print(sample.to_string(index=False))

---
## 5. Export Results

In [ ]:
# Merge clusters back to full dataset
result = df.merge(eligible[['app_id', 'ml_cluster']], on='app_id', how='left')

# Export
output_cols = ['app_id', 'name', 'price', 'genres', 'total_reviews',
               'satisfaction_pct', 'peak_ccu', 'release_year', 'ml_cluster']
output = result[[c for c in output_cols if c in result.columns]]

# Save to outputs/ if it exists, otherwise current folder
import os
out_dir = 'outputs' if os.path.isdir('outputs') else '../outputs' if os.path.isdir('../outputs') else '.'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'ml_clusters.csv')
output.to_csv(out_path, index=False)

print(f"✅ Saved: {out_path}")
print(f"   Games with cluster:    {output['ml_cluster'].notna().sum()}")
print(f"   Games without cluster: {output['ml_cluster'].isna().sum()}")

---
## 6. Launch Dashboard

Run the cell below to start the Dash dashboard. Click the link that appears to open it.

**Note:** The dashboard reads from the `outputs/` folder. Make sure `ml_clusters.csv` and `genre_price_summary.csv` are there.

In [ ]:
# Option A: Run dashboard inline in Jupyter (JupyterDash)
# If this doesn't work, use Option B below.
try:
    from jupyter_dash import JupyterDash
    JUPYTER_MODE = True
except ImportError:
    JUPYTER_MODE = False
    print("jupyter-dash not installed. Use Option B: run dashboard.py from terminal.")
    print("  → Open a terminal and run: python scripts/dashboard.py")
    print("  → Then open http://127.0.0.1:8050")

In [ ]:
# Option B: Run the dashboard from terminal
# Uncomment and run this cell if you prefer:
# !python scripts/dashboard.py

# Or if you're in this delivery folder:
# !python dashboard.py

print("To launch the dashboard:")
print("  1. Open a terminal")
print("  2. cd into the project folder")
print("  3. Run: python scripts/dashboard.py")
print("  4. Open http://127.0.0.1:8050 in your browser")

---
## 📝 Documentation: Model Choices & Limitations

### Algorithm: K-Means Clustering
- **Why K-Means?** Simple, fast, interpretable — ideal for a first segmentation.
- **Features:** price, total_reviews (log), satisfaction_pct, peak_ccu (log)
- **Preprocessing:** StandardScaler (zero mean, unit variance); log transform on skewed features
- **Cluster selection:** Silhouette score (automated)

### Limitations
- Games without reviews are excluded (no satisfaction data)
- K-Means assumes spherical clusters — non-linear patterns may be missed
- The sample dataset (90 games) is small; results will improve with the full dataset
- `average_playtime_minutes` was excluded (all zeros in sample)

### Business Value
- Identifies market segments: premium/popular vs budget/niche
- Helps indie developers position their games
- Combined with Hidden Gems and genre analysis, provides a full market picture